# Task 2 - Cross-Attention Visualization for Entailment Reasoning

This notebook:
- adds a fusion attention stack between image patches and text tokens
- captures attention weights from each fusion layer with `register_forward_hook()`
- selects 20 examples across entailment/neutral/contradiction with easy/medium/hard difficulty
- saves attention tensors (`[num_heads, seq_len_image, seq_len_text]`)
- creates token-level overlay heatmaps and class-wise attention galleries

Outputs are written to `task2_artifacts/`.


In [ ]:
# Optional installs if needed.
# %pip install torch torchvision transformers pandas pillow matplotlib


In [ ]:
from task2_cross_attention_viz import Task2Config, run_task2

config = Task2Config(
    test_csv='cleaned_snli_ve_test.csv',
    checkpoint_path='final_sota_visual_entailment3.pth',
    output_dir='task2_artifacts',
    num_examples=20,
    pool_size=2500,
    batch_size=8,
    max_text_length=48,
    fusion_attention_layers=2,
    fusion_attention_heads=8,
    top_tokens_per_example=4,
)
config


In [ ]:
summary = run_task2(config)
summary


In [ ]:
import json
import pandas as pd
from pathlib import Path

out_dir = Path(config.output_dir)
with (out_dir / 'task2_summary.json').open() as f:
    summary = json.load(f)

print('Selected examples:', summary['num_selected_examples'])
print('Label counts:', summary['selected_label_counts'])
print('Difficulty counts:', summary['selected_difficulty_counts'])

selected = pd.read_csv(out_dir / 'task2_selected_examples.csv')
selected.head(20)


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

out_dir = Path(config.output_dir)

gallery_paths = [
    out_dir / 'class_gallery_entailment.png',
    out_dir / 'class_gallery_neutral.png',
    out_dir / 'class_gallery_contradiction.png',
]

for p in gallery_paths:
    if p.exists():
        plt.figure(figsize=(12, 6))
        plt.imshow(Image.open(p))
        plt.title(p.name)
        plt.axis('off')
        plt.show()
